# M1 — Lexical retrieval: E01–E04

**Goal.** Establish and strengthen a reproducible high-recall lexical baseline for Avito service candidate generation under the frozen `benchmark_aligned_proxy_v1` protocol from M0.

**Experiments.**

- **E01:** BM25 over title + parameters + description;
- **E02:** field and query-representation ablations;
- **E03:** title character n-gram TF-IDF and reciprocal-rank fusion (RRF);
- **E04:** Russian text-preprocessing ablation for BM25: control, stemming, lemmatization, stop words, and their combinations.

All retrieval uses the M0 category partition: `item_category_id == search_category`. The notebook logs aggregate technical and quality metrics to a live ClearML task; it does not upload source texts or Parquet files.

## Plan

1. **Reproducibility and ClearML** — load local configuration and create a live Task without exposing secrets.
2. **Frozen proxy and category partition** — recreate the M0 split and the allowed corpus for each query.
3. **Text preprocessing and exact BM25** — define six Russian-safe transformations and verify the sparse BM25 implementation.
4. **E01–E02** — run the original BM25 baseline and controlled field/query ablations.
5. **E03** — add title character TF-IDF and deterministic RRF fusion.
6. **E04** — compare stemming, lemmatization and stop-word combinations; test the winning word-BM25 preprocessing in the final fusion.
7. **Persist selected texts** — write the selected BM25 and char-TF-IDF representations to `artifacts/text_preprocessing/m1_lexical_best_v1/`.
8. **Results and decision** — compare macro Recall@50, runtime and resource use; log the evidence to ClearML.

The preprocessing winner is selected on macro Recall@50, breaking ties by Recall@200 and then by deterministic configuration order. A material final-pipeline gain should be rechecked at a later hybrid stage before it replaces the frozen baseline.

## 1. Reproducibility and live ClearML

The notebook loads ClearML settings from the repository-local `.env` *before* importing ClearML. It requires a live configured task and deliberately has no offline fallback. No credential value is printed or persisted in notebook output.

In [1]:
from __future__ import annotations

import gc
import json
import os
import re
import resource
import sys
import time
from collections import Counter
from functools import lru_cache, partial
from pathlib import Path

import numpy as np
import pandas as pd
import pymorphy3
import snowballstemmer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from stop_words import get_stop_words

SEED = 42
np.random.seed(SEED)

REPO_ROOT = Path.cwd()
DATA_DIR = Path(os.environ.get("AVITO_DATA_DIR", "/Users/kite/Downloads/dataset"))
TRAIN_PATH = DATA_DIR / "train.parquet"
BENCHMARK_QUERIES_PATH = DATA_DIR / "benchmark_queries.parquet"
BENCHMARK_ITEMS_PATH = DATA_DIR / "benchmark_items.parquet"
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "text_preprocessing" / "m1_lexical_best_v1"

TOP_K = 200
METRIC_KS = (1, 5, 10, 20, 50, 200)
BM25_K1 = 1.5
BM25_B = 0.75
CHAR_MAX_FEATURES = 200_000
CHAR_BATCH_SIZE = 32
PREPROCESSING_VERSION = "m1_lexical_best_v1"
PREPROCESSING_ORDER = (
    "control",
    "stem",
    "lemma",
    "stopwords",
    "stem_stopwords",
    "lemma_stopwords",
)

assert REPO_ROOT.joinpath(".env").exists(), "Create and fill .env before running M1."
assert all(path.exists() for path in (TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH))

NOTEBOOK_STARTED = time.perf_counter()


def load_dotenv(path: Path) -> None:
    """Minimal .env loader; never prints values and does not overwrite shell env."""
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


load_dotenv(REPO_ROOT / ".env")
required_clearml = ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY")
missing_clearml = [key for key in required_clearml if not os.environ.get(key)]
if missing_clearml:
    raise RuntimeError(f"Missing ClearML values in .env: {', '.join(missing_clearml)}")
if os.environ.get("CLEARML_OFFLINE_MODE", "").lower() in {"1", "true", "yes"}:
    raise RuntimeError("M1 requires a live ClearML task; CLEARML_OFFLINE_MODE must not be enabled.")

print({
    "seed": SEED,
    "data_dir": str(DATA_DIR),
    "artifact_dir": str(ARTIFACT_DIR),
    "top_k": TOP_K,
    "bm25": {"k1": BM25_K1, "b": BM25_B},
    "char_tfidf_max_features": CHAR_MAX_FEATURES,
    "preprocessing_variants": list(PREPROCESSING_ORDER),
    "clearml_credentials_present": True,
})

{'seed': 42, 'data_dir': '/Users/kite/Downloads/dataset', 'artifact_dir': '/Users/kite/Documents/Карьера/avito-services-candidate-generation/artifacts/text_preprocessing/m1_lexical_best_v1', 'top_k': 200, 'bm25': {'k1': 1.5, 'b': 0.75}, 'char_tfidf_max_features': 200000, 'preprocessing_variants': ['control', 'stem', 'lemma', 'stopwords', 'stem_stopwords', 'lemma_stopwords'], 'clearml_credentials_present': True}


In [2]:
# Import happens only after the .env credentials are available.
from clearml import Task

CLEARML_PROJECT = "avito-retrieval"
CLEARML_TASK_NAME = "E01-E04__lexical-preprocessing__bm25-tfidf__s42"

clearml_task = Task.init(
    project_name=CLEARML_PROJECT,
    task_name=CLEARML_TASK_NAME,
    reuse_last_task_id=False,
    auto_connect_frameworks=False,
)
clearml_task.connect(
    {
        "stage": "M1_lexical_retrieval_preprocessing",
        "validation_protocol": "benchmark_aligned_proxy_v1",
        "seed": SEED,
        "top_k": TOP_K,
        "bm25_k1": BM25_K1,
        "bm25_b": BM25_B,
        "category_partition": "item_category_id == search_category; full-corpus fallback when no matching corpus partition exists",
        "char_tfidf": {"analyzer": "char_wb", "ngram_range": [3, 5], "max_features": CHAR_MAX_FEATURES},
        "preprocessing_variants": list(PREPROCESSING_ORDER),
        "artifact_dir": str(ARTIFACT_DIR),
    },
    name="config",
)
clearml_logger = clearml_task.get_logger()
print({"clearml_project": CLEARML_PROJECT, "clearml_task_id": clearml_task.id, "offline_mode": False})

ClearML Task: created new task id=40d91857fb7b4382a64f32561e716d42


2026-09-18 15:52:24,935 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []


ClearML results page: https://app.clear.ml/projects/588424e922a44a95aa934ad17ca57931/tasks/40d91857fb7b4382a64f32561e716d42/output/log
2026-09-18 15:52:26,562 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found


ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


{'clearml_project': 'avito-retrieval', 'clearml_task_id': '40d91857fb7b4382a64f32561e716d42', 'offline_mode': False}


## 2. Recreate the frozen M0 proxy exactly

`query_group` is built from all search-side fields over the concatenation of train and benchmark query contexts. The split is then applied to the benchmark-corpus-aligned train positive rows, exactly as in M0.

In [3]:
SEARCH_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
TRAIN_COLUMNS = [*SEARCH_COLUMNS, "item_id"]
BENCHMARK_QUERY_COLUMNS = ["query_id", *SEARCH_COLUMNS]
ITEM_COLUMNS = [
    "item_id",
    "item_title_raw",
    "item_description_raw",
    "item_infm_params_text",
    "item_category_id",
]

load_started = time.perf_counter()
train_pairs = pd.read_parquet(TRAIN_PATH, columns=TRAIN_COLUMNS)
benchmark_queries = pd.read_parquet(BENCHMARK_QUERIES_PATH, columns=BENCHMARK_QUERY_COLUMNS)
benchmark_items = pd.read_parquet(BENCHMARK_ITEMS_PATH, columns=ITEM_COLUMNS)
load_seconds = time.perf_counter() - load_started


def canonical_query_frame(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame[SEARCH_COLUMNS].copy()
    for column in ("search_query", "search_infm_params_text"):
        result[column] = (
            result[column]
            .astype("string")
            .fillna("<NA>")
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )
    for column in ("search_location_id", "search_is_delivery_search", "search_category"):
        result[column] = result[column].astype("string").fillna("<NA>")
    return result

all_contexts = pd.concat(
    [canonical_query_frame(train_pairs), canonical_query_frame(benchmark_queries)],
    ignore_index=True,
)
all_group_ids, _ = pd.factorize(pd.MultiIndex.from_frame(all_contexts), sort=False)
train_pairs["query_group"] = all_group_ids[: len(train_pairs)]

benchmark_item_id_set = set(benchmark_items["item_id"].astype(str))
proxy_pairs = train_pairs.loc[train_pairs["item_id"].astype(str).isin(benchmark_item_id_set)].copy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
proxy_train_idx, proxy_valid_idx = next(splitter.split(proxy_pairs, groups=proxy_pairs["query_group"]))
proxy_train_pairs = proxy_pairs.iloc[proxy_train_idx].copy()
proxy_valid_pairs = proxy_pairs.iloc[proxy_valid_idx].copy()
assert set(proxy_train_pairs["query_group"]).isdisjoint(set(proxy_valid_pairs["query_group"]))

validation_queries = (
    proxy_valid_pairs.sort_values("query_group")
    .drop_duplicates("query_group")
    .loc[:, ["query_group", *SEARCH_COLUMNS]]
    .reset_index(drop=True)
)
gold_by_group = (
    proxy_valid_pairs.groupby("query_group", sort=False)["item_id"]
    .agg(lambda values: frozenset(values.astype(str)))
    .to_dict()
)
gold_sets = [gold_by_group[group] for group in validation_queries["query_group"]]

proxy_summary = pd.DataFrame(
    [
        {"partition": "train", "positive_rows": len(proxy_train_pairs), "query_groups": proxy_train_pairs["query_group"].nunique()},
        {"partition": "validation", "positive_rows": len(proxy_valid_pairs), "query_groups": proxy_valid_pairs["query_group"].nunique()},
    ]
)
display(proxy_summary)
print({
    "load_seconds": round(load_seconds, 2),
    "proxy_positive_rows": len(proxy_pairs),
    "validation_unique_queries": len(validation_queries),
    "validation_gold_cardinality_median": float(np.median([len(gold) for gold in gold_sets])),
})

,partition,positive_rows,query_groups
0,train,26377,21239
1,validation,6633,5310


{'load_seconds': 1.34, 'proxy_positive_rows': 33010, 'validation_unique_queries': 5310, 'validation_gold_cardinality_median': 1.0}


In [4]:
# M0 hard partition, with the documented fallback if a category has no corpus rows.
# In this split, 5,309 queries have category 114; one query has category 0,
# which has no matching item_category_id and therefore uses the full corpus.
candidate_items = benchmark_items.reset_index(drop=True)
candidate_item_ids = candidate_items["item_id"].astype(str).to_numpy()
all_candidate_indices = np.arange(len(candidate_items), dtype=np.int64)
category_to_indices = {
    category: group.index.to_numpy(dtype=np.int64)
    for category, group in candidate_items.groupby("item_category_id", sort=False)
}


def allowed_indices_for_category(category: object) -> tuple[np.ndarray, bool]:
    indices = category_to_indices.get(category)
    if indices is None or len(indices) == 0:
        return all_candidate_indices, True
    return indices, False


allowed_indices_by_query = []
used_full_corpus_fallback = []
for category in validation_queries["search_category"]:
    allowed, fallback = allowed_indices_for_category(category)
    allowed_indices_by_query.append(allowed)
    used_full_corpus_fallback.append(fallback)

category_oracle_recall = float(
    np.mean([
        len(gold & set(candidate_item_ids[allowed])) / len(gold)
        for gold, allowed in zip(gold_sets, allowed_indices_by_query, strict=True)
    ])
)
assert category_oracle_recall == 1.0, "The category rule plus fallback removes a validation positive."

print({
    "validation_categories": validation_queries["search_category"].value_counts().to_dict(),
    "items_in_category_114": int((candidate_items["item_category_id"] == 114).sum()),
    "full_benchmark_items": len(candidate_items),
    "full_corpus_fallback_queries": int(sum(used_full_corpus_fallback)),
    "validation_category_oracle_recall@50": category_oracle_recall,
})

{'validation_categories': {114: 5309, 0: 1}, 'items_in_category_114': 187336, 'full_benchmark_items': 189212, 'full_corpus_fallback_queries': 1, 'validation_category_oracle_recall@50': 1.0}


## 3. Text preprocessing and exact sparse BM25

Every transformation starts from the same control normalization: lower case, `ё → е`, non-alphanumeric characters replaced by spaces, and collapsed whitespace. E04 then applies one of: Russian Snowball stemming, `pymorphy3` lemmatization, the `stop-words` Russian list, or a combination. Stop words are removed before morphology.

The sparse implementation is validated against `rank_bm25.BM25Okapi` on a deterministic example. All preprocessing is performed locally; no external API is used.

In [5]:
TOKEN_PATTERN = r"(?u)\b[0-9a-zа-я]{2,}\b"
NON_WORD_RE = re.compile(r"[^0-9a-zа-я]+")


def normalize_russian_text(value: object) -> str:
    """M1 control transformation shared by all preprocessing variants."""
    text = "" if pd.isna(value) else str(value)
    text = text.lower().replace("ё", "е")
    text = NON_WORD_RE.sub(" ", text)
    return " ".join(text.split())


RUSSIAN_STOPWORDS = frozenset(
    token
    for word in get_stop_words("ru")
    for token in normalize_russian_text(word).split()
)
STEMMER = snowballstemmer.stemmer("russian")
MORPH = pymorphy3.MorphAnalyzer()

PREPROCESSING_SPECS = {
    "control": {"morphology": "none", "remove_stopwords": False},
    "stem": {"morphology": "snowball_russian", "remove_stopwords": False},
    "lemma": {"morphology": "pymorphy3_normal_form", "remove_stopwords": False},
    "stopwords": {"morphology": "none", "remove_stopwords": True},
    "stem_stopwords": {"morphology": "snowball_russian", "remove_stopwords": True},
    "lemma_stopwords": {"morphology": "pymorphy3_normal_form", "remove_stopwords": True},
}
assert tuple(PREPROCESSING_SPECS) == PREPROCESSING_ORDER


@lru_cache(maxsize=1_000_000)
def stem_token(token: str) -> str:
    return STEMMER.stemWord(token)


@lru_cache(maxsize=1_000_000)
def lemma_token(token: str) -> str:
    parsed = MORPH.parse(token)
    return parsed[0].normal_form if parsed else token


def preprocess_russian_text(value: object, preprocessing_key: str = "control") -> str:
    """Apply one deterministic E04 configuration to a text value."""
    spec = PREPROCESSING_SPECS[preprocessing_key]
    tokens = normalize_russian_text(value).split()
    if spec["remove_stopwords"]:
        tokens = [token for token in tokens if token not in RUSSIAN_STOPWORDS]
    if spec["morphology"] == "snowball_russian":
        tokens = [stem_token(token) for token in tokens]
    elif spec["morphology"] == "pymorphy3_normal_form":
        tokens = [lemma_token(token) for token in tokens]
    return " ".join(tokens)


class SparseBM25:
    """Memory-efficient exact Okapi BM25 over a CountVectorizer CSC matrix."""

    def __init__(self, preprocessing_key: str = "control", k1: float = 1.5, b: float = 0.75, epsilon: float = 0.25):
        self.preprocessing_key = preprocessing_key
        self.k1 = k1
        self.b = b
        self.epsilon = epsilon

    def fit(self, documents: list[str]) -> "SparseBM25":
        self.vectorizer = CountVectorizer(
            preprocessor=partial(preprocess_russian_text, preprocessing_key=self.preprocessing_key),
            token_pattern=TOKEN_PATTERN,
            lowercase=False,
            dtype=np.float32,
        )
        counts_csr = self.vectorizer.fit_transform(documents)
        self.doc_len = np.asarray(counts_csr.sum(axis=1)).ravel().astype(np.float32)
        self.n_docs = counts_csr.shape[0]
        self.avgdl = float(self.doc_len.mean())
        self.matrix = counts_csr.tocsc()
        del counts_csr

        document_frequency = np.diff(self.matrix.indptr).astype(np.float64)
        idf = np.log((self.n_docs - document_frequency + 0.5) / (document_frequency + 0.5))
        average_idf = float(idf.mean())
        idf[idf < 0] = self.epsilon * average_idf
        self.idf = idf.astype(np.float32)
        self.norm = self.k1 * (1.0 - self.b + self.b * self.doc_len / self.avgdl)
        self.analyzer = self.vectorizer.build_analyzer()
        return self

    @property
    def n_features(self) -> int:
        return len(self.vectorizer.vocabulary_)

    def score(self, query: str) -> np.ndarray:
        scores = np.zeros(self.n_docs, dtype=np.float32)
        for token, query_tf in Counter(self.analyzer(query)).items():
            feature_idx = self.vectorizer.vocabulary_.get(token)
            if feature_idx is None:
                continue
            start, stop = self.matrix.indptr[feature_idx : feature_idx + 2]
            rows = self.matrix.indices[start:stop]
            term_tf = self.matrix.data[start:stop]
            scores[rows] += query_tf * self.idf[feature_idx] * (
                term_tf * (self.k1 + 1.0) / (term_tf + self.norm[rows])
            )
        return scores

    def top_k(self, query: str, k: int, allowed_indices: np.ndarray | None = None) -> np.ndarray:
        scores = self.score(query)
        allowed = np.arange(self.n_docs, dtype=np.int64) if allowed_indices is None else allowed_indices
        k = min(k, len(allowed))
        eligible_scores = scores[allowed]
        selected_positions = np.argpartition(eligible_scores, len(allowed) - k)[len(allowed) - k :]
        selected_positions = selected_positions[np.argsort(eligible_scores[selected_positions])[::-1]]
        return allowed[selected_positions]


# Correctness check against the open-source reference implementation.
toy_documents = ["ремонт ноутбука", "ремонт телевизора", "установка двери"]
toy_query = "ремонт"
reference_scores = BM25Okapi([document.split() for document in toy_documents], k1=BM25_K1, b=BM25_B).get_scores(toy_query.split())
checked_scores = SparseBM25(k1=BM25_K1, b=BM25_B).fit(toy_documents).score(toy_query)
np.testing.assert_allclose(checked_scores, reference_scores, rtol=1e-6, atol=1e-6)
assert preprocess_russian_text("Ёлки, и баня!", "control") == "елки и баня"
assert preprocess_russian_text("Ёлки, и баня!", "stopwords") == "елки баня"
print({"bm25_reference_check": "passed", "russian_stopwords": len(RUSSIAN_STOPWORDS), "preprocessing_configs": PREPROCESSING_SPECS})

{'bm25_reference_check': 'passed', 'russian_stopwords': 417, 'preprocessing_configs': {'control': {'morphology': 'none', 'remove_stopwords': False}, 'stem': {'morphology': 'snowball_russian', 'remove_stopwords': False}, 'lemma': {'morphology': 'pymorphy3_normal_form', 'remove_stopwords': False}, 'stopwords': {'morphology': 'none', 'remove_stopwords': True}, 'stem_stopwords': {'morphology': 'snowball_russian', 'remove_stopwords': True}, 'lemma_stopwords': {'morphology': 'pymorphy3_normal_form', 'remove_stopwords': True}}}


In [6]:
def compose_document_text(frame: pd.DataFrame, fields: tuple[str, ...]) -> list[str]:
    text = frame[fields[0]].fillna("").astype(str)
    for field in fields[1:]:
        text = text.str.cat(frame[field].fillna("").astype(str), sep=" ")
    return text.tolist()


def retrieve_bm25(index: SparseBM25, queries: list[str], allowed_by_query: list[np.ndarray], top_k: int) -> tuple[list[np.ndarray], float]:
    started = time.perf_counter()
    rankings = [
        index.top_k(query, top_k, allowed)
        for query, allowed in zip(queries, allowed_by_query, strict=True)
    ]
    return rankings, time.perf_counter() - started


def top_k_from_scores(scores: np.ndarray, k: int) -> np.ndarray:
    k = min(k, len(scores))
    selected = np.argpartition(scores, len(scores) - k)[len(scores) - k :]
    return selected[np.argsort(scores[selected])[::-1]]


def macro_recall(rankings: list[np.ndarray], gold: list[frozenset[str]], k: int) -> float:
    recalls = []
    for ranking, relevant in zip(rankings, gold, strict=True):
        predicted = set(candidate_item_ids[ranking[:k]])
        recalls.append(len(predicted & relevant) / len(relevant))
    return float(np.mean(recalls))


def hit_rate(rankings: list[np.ndarray], gold: list[frozenset[str]], k: int) -> float:
    hits = []
    for ranking, relevant in zip(rankings, gold, strict=True):
        hits.append(bool(set(candidate_item_ids[ranking[:k]]) & relevant))
    return float(np.mean(hits))


def peak_rss_mb() -> float:
    max_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return float(max_rss / (1024 * 1024 if sys.platform == "darwin" else 1024))


def evaluate(experiment: str, rankings: list[np.ndarray], *, index_seconds: float, retrieval_seconds: float,
             n_features: int, representation: str, metadata: dict[str, object] | None = None) -> dict[str, object]:
    record: dict[str, object] = {
        "experiment": experiment,
        "representation": representation,
        "index_seconds": index_seconds,
        "retrieval_seconds": retrieval_seconds,
        "total_seconds": index_seconds + retrieval_seconds,
        "features": n_features,
        "peak_rss_mb": peak_rss_mb(),
    }
    for k in METRIC_KS:
        record[f"recall@{k}"] = macro_recall(rankings, gold_sets, k)
    record["hit_rate@50"] = hit_rate(rankings, gold_sets, 50)
    if metadata:
        record.update(metadata)
    return record


def fit_sparse_bm25(documents: list[str], preprocessing_key: str = "control") -> tuple[SparseBM25, float]:
    started = time.perf_counter()
    index = SparseBM25(preprocessing_key=preprocessing_key, k1=BM25_K1, b=BM25_B).fit(documents)
    return index, time.perf_counter() - started


def fit_and_retrieve_char_tfidf(
    documents: list[str], queries: list[str], allowed_by_query: list[np.ndarray], preprocessing_key: str,
) -> tuple[TfidfVectorizer, list[np.ndarray], float, float]:
    """Batch sparse cosine retrieval to make the preprocessing comparison feasible."""
    fit_started = time.perf_counter()
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        preprocessor=partial(preprocess_russian_text, preprocessing_key=preprocessing_key),
        lowercase=False,
        ngram_range=(3, 5),
        min_df=2,
        max_features=CHAR_MAX_FEATURES,
        sublinear_tf=True,
        dtype=np.float32,
    )
    matrix = vectorizer.fit_transform(documents)
    index_seconds = time.perf_counter() - fit_started

    retrieval_started = time.perf_counter()
    query_matrix = vectorizer.transform(queries)
    rankings: list[np.ndarray] = []
    for start in range(0, query_matrix.shape[0], CHAR_BATCH_SIZE):
        stop = min(start + CHAR_BATCH_SIZE, query_matrix.shape[0])
        batch_scores = (query_matrix[start:stop] @ matrix.T).toarray()
        for row_scores, allowed in zip(batch_scores, allowed_by_query[start:stop], strict=True):
            selected_positions = top_k_from_scores(row_scores[allowed], TOP_K)
            rankings.append(allowed[selected_positions])
    retrieval_seconds = time.perf_counter() - retrieval_started
    return vectorizer, rankings, index_seconds, retrieval_seconds


def rrf_fuse(rankings_by_source: list[np.ndarray], top_k: int, rrf_k: int = 60) -> np.ndarray:
    scores: dict[int, float] = {}
    for rankings in rankings_by_source:
        for rank, item_idx in enumerate(rankings, start=1):
            scores[int(item_idx)] = scores.get(int(item_idx), 0.0) + 1.0 / (rrf_k + rank)
    ordered = sorted(scores, key=lambda item_idx: (-scores[item_idx], item_idx))
    return np.asarray(ordered[:top_k], dtype=np.int64)


query_text = validation_queries["search_query"].fillna("").astype(str).tolist()
query_with_filters = (
    validation_queries["search_query"].fillna("").astype(str)
    + " "
    + validation_queries["search_infm_params_text"].fillna("").astype(str)
).tolist()

assert len(query_text) == len(gold_sets) == 5310
results: list[dict[str, object]] = []
print({"validation_queries": len(query_text), "retrieval_top_k": TOP_K, "char_batch_size": CHAR_BATCH_SIZE})

{'validation_queries': 5310, 'retrieval_top_k': 200, 'char_batch_size': 32}


## 4. E01 — Plain BM25

The prescribed first baseline indexes the concatenation of title, parameters and description; the query contains only `search_query`.

In [7]:
all_fields = ("item_title_raw", "item_infm_params_text", "item_description_raw")
all_documents = compose_document_text(candidate_items, all_fields)
all_bm25, all_index_seconds = fit_sparse_bm25(all_documents, preprocessing_key="control")
rankings_all, all_retrieval_seconds = retrieve_bm25(all_bm25, query_text, allowed_indices_by_query, TOP_K)
results.append(evaluate(
    "E01_bm25_title_params_description__query",
    rankings_all,
    index_seconds=all_index_seconds,
    retrieval_seconds=all_retrieval_seconds,
    n_features=all_bm25.n_features,
    representation="title + parameters + description | query",
    metadata={"bm25_preprocessing": "control", "char_preprocessing": "not_applicable"},
))
print(results[-1])

{'experiment': 'E01_bm25_title_params_description__query', 'representation': 'title + parameters + description | query', 'index_seconds': 46.25104741699988, 'retrieval_seconds': 7.3913719160000255, 'total_seconds': 53.64241933299991, 'features': 595180, 'peak_rss_mb': 2773.0, 'recall@1': 0.019350282485875708, 'recall@5': 0.07426867545511613, 'recall@10': 0.11671971123666039, 'recall@20': 0.1679728126027561, 'recall@50': 0.2840713388933728, 'recall@200': 0.5145393238274594, 'hit_rate@50': 0.29397363465160076, 'bm25_preprocessing': 'control', 'char_preprocessing': 'not_applicable'}


## 5. E02 — Controlled lexical ablations

Only one textual representation changes at a time. The all-fields index is reused for the query-filter ablation, so its retrieval timing excludes a duplicated index build.

In [8]:
# E02a: title only.
title_documents = compose_document_text(candidate_items, ("item_title_raw",))
title_bm25, title_index_seconds = fit_sparse_bm25(title_documents, preprocessing_key="control")
rankings_title, title_retrieval_seconds = retrieve_bm25(title_bm25, query_text, allowed_indices_by_query, TOP_K)
results.append(evaluate(
    "E02a_bm25_title__query",
    rankings_title,
    index_seconds=title_index_seconds,
    retrieval_seconds=title_retrieval_seconds,
    n_features=title_bm25.n_features,
    representation="title | query",
    metadata={"bm25_preprocessing": "control", "char_preprocessing": "not_applicable"},
))
print(results[-1])
del title_bm25, title_documents
gc.collect()

# E02b: title + item parameters.
title_params_documents = compose_document_text(candidate_items, ("item_title_raw", "item_infm_params_text"))
title_params_bm25, title_params_index_seconds = fit_sparse_bm25(title_params_documents, preprocessing_key="control")
rankings_title_params, title_params_retrieval_seconds = retrieve_bm25(title_params_bm25, query_text, allowed_indices_by_query, TOP_K)
results.append(evaluate(
    "E02b_bm25_title_params__query",
    rankings_title_params,
    index_seconds=title_params_index_seconds,
    retrieval_seconds=title_params_retrieval_seconds,
    n_features=title_params_bm25.n_features,
    representation="title + parameters | query",
    metadata={"bm25_preprocessing": "control", "char_preprocessing": "not_applicable"},
))
print(results[-1])
del title_params_bm25, title_params_documents
gc.collect()

# E02c: add the search filter text to the E01 query representation.
rankings_all_filters, all_filters_retrieval_seconds = retrieve_bm25(all_bm25, query_with_filters, allowed_indices_by_query, TOP_K)
results.append(evaluate(
    "E02c_bm25_title_params_description__query_filters",
    rankings_all_filters,
    index_seconds=0.0,
    retrieval_seconds=all_filters_retrieval_seconds,
    n_features=all_bm25.n_features,
    representation="title + parameters + description | query + search filters (reused E01 index)",
    metadata={"bm25_preprocessing": "control", "char_preprocessing": "not_applicable"},
))
print(results[-1])

{'experiment': 'E02a_bm25_title__query', 'representation': 'title | query', 'index_seconds': 0.9795732920001683, 'retrieval_seconds': 5.037187708000147, 'total_seconds': 6.016761000000315, 'features': 42417, 'peak_rss_mb': 2773.0, 'recall@1': 0.015819209039548022, 'recall@5': 0.06155681104833648, 'recall@10': 0.09379912115505337, 'recall@20': 0.13725038113173707, 'recall@50': 0.22069043135144834, 'recall@200': 0.3856454279137895, 'hit_rate@50': 0.23107344632768362, 'bm25_preprocessing': 'control', 'char_preprocessing': 'not_applicable'}


{'experiment': 'E02b_bm25_title_params__query', 'representation': 'title + parameters | query', 'index_seconds': 18.753125833000013, 'retrieval_seconds': 5.949346208999941, 'total_seconds': 24.702472041999954, 'features': 126289, 'peak_rss_mb': 2773.0, 'recall@1': 0.01820464532328939, 'recall@5': 0.05619899560577526, 'recall@10': 0.09064344005021972, 'recall@20': 0.13950168893671716, 'recall@50': 0.21605536125310135, 'recall@200': 0.3954993274145816, 'hit_rate@50': 0.22523540489642185, 'bm25_preprocessing': 'control', 'char_preprocessing': 'not_applicable'}


ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


{'experiment': 'E02c_bm25_title_params_description__query_filters', 'representation': 'title + parameters + description | query + search filters (reused E01 index)', 'index_seconds': 0.0, 'retrieval_seconds': 21.59819483299998, 'total_seconds': 21.59819483299998, 'features': 595180, 'peak_rss_mb': 2773.0, 'recall@1': 0.01977401129943503, 'recall@5': 0.07343691148775894, 'recall@10': 0.11187696170747018, 'recall@20': 0.1684180790960452, 'recall@50': 0.278385570800825, 'recall@200': 0.5055377694078259, 'hit_rate@50': 0.2887005649717514, 'bm25_preprocessing': 'control', 'char_preprocessing': 'not_applicable'}


## 6. E03 — Strong lexical alternatives and fusion

The control char-TFIDF representation is retained as the reference because character n-grams already encode much of Russian inflectional similarity. Its batched implementation preserves exact cosine scores while avoiding a Python-level transform loop for every query.

In [9]:
# E03a: char n-gram TF-IDF on short titles with control preprocessing.
char_documents = compose_document_text(candidate_items, ("item_title_raw",))
char_vectorizer, rankings_char, char_index_seconds, char_retrieval_seconds = fit_and_retrieve_char_tfidf(
    char_documents, query_text, allowed_indices_by_query, preprocessing_key="control"
)
results.append(evaluate(
    "E03a_char_tfidf_title__query",
    rankings_char,
    index_seconds=char_index_seconds,
    retrieval_seconds=char_retrieval_seconds,
    n_features=len(char_vectorizer.vocabulary_),
    representation="char_wb 3–5 grams: title | query",
    metadata={"bm25_preprocessing": "not_applicable", "char_preprocessing": "control"},
))
print(results[-1])

# E03b: reciprocal-rank fusion of title-only and E01 all-fields BM25.
rrf_bm25_started = time.perf_counter()
rankings_rrf_bm25 = [rrf_fuse([title_ranking, all_ranking], TOP_K) for title_ranking, all_ranking in zip(rankings_title, rankings_all, strict=True)]
rrf_bm25_seconds = time.perf_counter() - rrf_bm25_started
results.append(evaluate(
    "E03b_rrf_bm25_title_all_fields",
    rankings_rrf_bm25,
    index_seconds=0.0,
    retrieval_seconds=rrf_bm25_seconds,
    n_features=0,
    representation="RRF(title-BM25, all-fields-BM25); reused indices",
    metadata={"bm25_preprocessing": "control", "char_preprocessing": "not_applicable"},
))
print(results[-1])

# E03c: fuse the strongest broad BM25 source with char TF-IDF.
rrf_char_started = time.perf_counter()
rankings_rrf_char = [rrf_fuse([all_ranking, char_ranking], TOP_K) for all_ranking, char_ranking in zip(rankings_all, rankings_char, strict=True)]
rrf_char_seconds = time.perf_counter() - rrf_char_started
results.append(evaluate(
    "E03c_rrf_bm25_all_fields_char_tfidf",
    rankings_rrf_char,
    index_seconds=0.0,
    retrieval_seconds=rrf_char_seconds,
    n_features=0,
    representation="RRF(all-fields-BM25, title-char-TFIDF); reused indices",
    metadata={"bm25_preprocessing": "control", "char_preprocessing": "control"},
))
print(results[-1])

{'experiment': 'E03a_char_tfidf_title__query', 'representation': 'char_wb 3–5 grams: title | query', 'index_seconds': 6.636439834000157, 'retrieval_seconds': 33.03241170900037, 'total_seconds': 39.66885154300053, 'features': 97143, 'peak_rss_mb': 2773.0, 'recall@1': 0.017608286252354048, 'recall@5': 0.058857501569365984, 'recall@10': 0.09303515379786566, 'recall@20': 0.1392467043314501, 'recall@50': 0.2364576271186441, 'recall@200': 0.43353261291961853, 'hit_rate@50': 0.24538606403013183, 'bm25_preprocessing': 'not_applicable', 'char_preprocessing': 'control'}


{'experiment': 'E03b_rrf_bm25_title_all_fields', 'representation': 'RRF(title-BM25, all-fields-BM25); reused indices', 'index_seconds': 0.0, 'retrieval_seconds': 1.1044844580001154, 'total_seconds': 1.1044844580001154, 'features': 0, 'peak_rss_mb': 2773.0, 'recall@1': 0.020433145009416197, 'recall@5': 0.07228311362209668, 'recall@10': 0.11021155053358443, 'recall@20': 0.1688932831136221, 'recall@50': 0.283104370310585, 'recall@200': 0.5179554300062774, 'hit_rate@50': 0.29453860640301316, 'bm25_preprocessing': 'control', 'char_preprocessing': 'not_applicable'}


{'experiment': 'E03c_rrf_bm25_all_fields_char_tfidf', 'representation': 'RRF(all-fields-BM25, title-char-TFIDF); reused indices', 'index_seconds': 0.0, 'retrieval_seconds': 1.0209516670001904, 'total_seconds': 1.0209516670001904, 'features': 0, 'peak_rss_mb': 2773.0, 'recall@1': 0.022598870056497175, 'recall@5': 0.07620841180163214, 'recall@10': 0.11653797865662271, 'recall@20': 0.18228679042238363, 'recall@50': 0.2972636684303351, 'recall@200': 0.5344497354497354, 'hit_rate@50': 0.30847457627118646, 'bm25_preprocessing': 'control', 'char_preprocessing': 'control'}


## 7. E04 — Russian preprocessing ablation

The six configurations are compared first on the broad all-field BM25 source. Only the winner is then evaluated in the expensive char-TFIDF and RRF final-pipeline checks. This prevents us from selecting a morphology rule merely because it has a plausible linguistic motivation.

In [10]:
preprocessing_rows: list[dict[str, object]] = []
preprocessing_rankings: dict[str, list[np.ndarray]] = {}

# E01/E03 already produced the control rankings. Their fitted matrices are no
# longer needed, so release them before sequentially fitting five new BM25 indices.
control_bm25_features = all_bm25.n_features
del all_bm25, char_vectorizer
gc.collect()

for preprocessing_key in PREPROCESSING_ORDER:
    if preprocessing_key == "control":
        rankings = rankings_all
        index_seconds = all_index_seconds
        retrieval_seconds = all_retrieval_seconds
        n_features = control_bm25_features
    else:
        index, index_seconds = fit_sparse_bm25(all_documents, preprocessing_key=preprocessing_key)
        rankings, retrieval_seconds = retrieve_bm25(index, query_text, allowed_indices_by_query, TOP_K)
        n_features = index.n_features

    record = evaluate(
        f"E04_bm25_all_fields__{preprocessing_key}",
        rankings,
        index_seconds=index_seconds,
        retrieval_seconds=retrieval_seconds,
        n_features=n_features,
        representation=f"title + parameters + description | query | preprocessing={preprocessing_key}",
        metadata={"bm25_preprocessing": preprocessing_key, "char_preprocessing": "not_applicable"},
    )
    results.append(record)
    preprocessing_rows.append(record)
    preprocessing_rankings[preprocessing_key] = rankings
    print({
        "preprocessing": preprocessing_key,
        "recall@50": round(float(record["recall@50"]), 6),
        "index_seconds": round(float(index_seconds), 2),
        "retrieval_seconds": round(float(retrieval_seconds), 2),
    })

    if preprocessing_key != "control":
        del index
        gc.collect()

preprocessing_frame = (
    pd.DataFrame(preprocessing_rows)
    .sort_values(["recall@50", "recall@200", "experiment"], ascending=[False, False, True])
    .reset_index(drop=True)
)
best_bm25_preprocessing = str(preprocessing_frame.iloc[0]["bm25_preprocessing"])
rankings_best_bm25 = preprocessing_rankings[best_bm25_preprocessing]
display(preprocessing_frame[["bm25_preprocessing", "recall@50", "recall@200", "hit_rate@50", "index_seconds", "retrieval_seconds", "features"]])
print({"best_bm25_preprocessing": best_bm25_preprocessing})

# Test whether the best word-level transformation helps the established char-TFIDF fusion.
rrf_best_bm25_started = time.perf_counter()
rankings_rrf_best_bm25_control_char = [
    rrf_fuse([bm25_ranking, char_ranking], TOP_K)
    for bm25_ranking, char_ranking in zip(rankings_best_bm25, rankings_char, strict=True)
]
rrf_best_bm25_control_char_seconds = time.perf_counter() - rrf_best_bm25_started
results.append(evaluate(
    "E04_rrf_best_bm25_control_char",
    rankings_rrf_best_bm25_control_char,
    index_seconds=0.0,
    retrieval_seconds=rrf_best_bm25_control_char_seconds,
    n_features=0,
    representation=f"RRF(best E04 BM25={best_bm25_preprocessing}, control title-char-TFIDF)",
    metadata={"bm25_preprocessing": best_bm25_preprocessing, "char_preprocessing": "control"},
))

# Also apply the same winning transformation to title char-TFIDF. If it loses,
# the final pipeline keeps the control char representation instead of forcing morphology.
if best_bm25_preprocessing == "control":
    rankings_best_char = rankings_char
    best_char_index_seconds = 0.0
    best_char_retrieval_seconds = 0.0
else:
    best_char_vectorizer, rankings_best_char, best_char_index_seconds, best_char_retrieval_seconds = fit_and_retrieve_char_tfidf(
        char_documents, query_text, allowed_indices_by_query, preprocessing_key=best_bm25_preprocessing
    )
    results.append(evaluate(
        f"E04_char_tfidf_title__{best_bm25_preprocessing}",
        rankings_best_char,
        index_seconds=best_char_index_seconds,
        retrieval_seconds=best_char_retrieval_seconds,
        n_features=len(best_char_vectorizer.vocabulary_),
        representation=f"char_wb 3–5 grams: title | query | preprocessing={best_bm25_preprocessing}",
        metadata={"bm25_preprocessing": "not_applicable", "char_preprocessing": best_bm25_preprocessing},
    ))
    del best_char_vectorizer
    gc.collect()

rrf_best_both_started = time.perf_counter()
rankings_rrf_best_both = [
    rrf_fuse([bm25_ranking, char_ranking], TOP_K)
    for bm25_ranking, char_ranking in zip(rankings_best_bm25, rankings_best_char, strict=True)
]
rrf_best_both_seconds = time.perf_counter() - rrf_best_both_started
results.append(evaluate(
    "E04_rrf_best_bm25_best_char",
    rankings_rrf_best_both,
    index_seconds=0.0,
    retrieval_seconds=rrf_best_both_seconds,
    n_features=0,
    representation=f"RRF(best E04 BM25={best_bm25_preprocessing}, char preprocessing={best_bm25_preprocessing})",
    metadata={"bm25_preprocessing": best_bm25_preprocessing, "char_preprocessing": best_bm25_preprocessing},
))

FINAL_PIPELINE_EXPERIMENTS = (
    "E03c_rrf_bm25_all_fields_char_tfidf",
    "E04_rrf_best_bm25_control_char",
    "E04_rrf_best_bm25_best_char",
)
final_candidates = [record for record in results if record["experiment"] in FINAL_PIPELINE_EXPERIMENTS]
best_final_result = sorted(
    final_candidates,
    key=lambda record: (-float(record["recall@50"]), -float(record["recall@200"]), str(record["experiment"])),
)[0]
print({
    "best_final_experiment": best_final_result["experiment"],
    "best_final_recall@50": round(float(best_final_result["recall@50"]), 6),
    "final_bm25_preprocessing": best_final_result["bm25_preprocessing"],
    "final_char_preprocessing": best_final_result["char_preprocessing"],
})

{'preprocessing': 'control', 'recall@50': 0.284071, 'index_seconds': 46.25, 'retrieval_seconds': 7.39}


{'preprocessing': 'stem', 'recall@50': 0.305952, 'index_seconds': 78.0, 'retrieval_seconds': 8.01}


{'preprocessing': 'lemma', 'recall@50': 0.297142, 'index_seconds': 98.94, 'retrieval_seconds': 7.59}


{'preprocessing': 'stopwords', 'recall@50': 0.283914, 'index_seconds': 46.77, 'retrieval_seconds': 5.83}


{'preprocessing': 'stem_stopwords', 'recall@50': 0.304867, 'index_seconds': 53.75, 'retrieval_seconds': 6.96}


{'preprocessing': 'lemma_stopwords', 'recall@50': 0.297664, 'index_seconds': 56.12, 'retrieval_seconds': 6.5}


,bm25_preprocessing,recall@50,recall@200,hit_rate@50,index_seconds,retrieval_seconds,features
0,stem,0.305952,0.561345,0.316384,77.999484,8.012227,387751
1,stem_stopwords,0.304867,0.561800,0.315631,53.752551,6.957173,387689
2,lemma_stopwords,0.297664,0.559289,0.308663,56.122796,6.502069,396606
3,lemma,0.297142,0.560843,0.308098,98.937251,7.586830,396741
4,control,0.284071,0.514539,0.293974,46.251047,7.391372,595180
5,stopwords,0.283914,0.516028,0.293785,46.770035,5.834754,594793


{'best_bm25_preprocessing': 'stem'}


{'best_final_experiment': 'E04_rrf_best_bm25_control_char', 'best_final_recall@50': 0.312988, 'final_bm25_preprocessing': 'stem', 'final_char_preprocessing': 'control'}


## 8. Persist selected text representations

The chosen representation is materialized once for future retrieval stages. The ignored local directory `artifacts/text_preprocessing/m1_lexical_best_v1/` contains field-level BM25 texts, title text for char-TFIDF, train query contexts, benchmark queries, a manifest and a validation-results table. It contains no labels beyond train query context and no credentials.

In [11]:
final_bm25_preprocessing = str(best_final_result["bm25_preprocessing"])
final_char_preprocessing = str(best_final_result["char_preprocessing"])
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def preprocess_series(series: pd.Series, preprocessing_key: str) -> pd.Series:
    return series.fillna("").astype(str).map(
        lambda value: preprocess_russian_text(value, preprocessing_key=preprocessing_key)
    )


artifact_started = time.perf_counter()

artifact_items = candidate_items.loc[:, ["item_id", "item_category_id"]].copy()
for source_column, target_column in (
    ("item_title_raw", "item_title_bm25"),
    ("item_infm_params_text", "item_params_bm25"),
    ("item_description_raw", "item_description_bm25"),
):
    artifact_items[target_column] = preprocess_series(candidate_items[source_column], final_bm25_preprocessing)
artifact_items["item_text_bm25"] = (
    artifact_items["item_title_bm25"] + " "
    + artifact_items["item_params_bm25"] + " "
    + artifact_items["item_description_bm25"]
).str.replace(r"\s+", " ", regex=True).str.strip()
artifact_items["item_title_char_tfidf"] = preprocess_series(candidate_items["item_title_raw"], final_char_preprocessing)

artifact_benchmark_queries = benchmark_queries.loc[:, ["query_id", "search_category"]].copy()
artifact_benchmark_queries["search_query_bm25"] = preprocess_series(
    benchmark_queries["search_query"], final_bm25_preprocessing
)
artifact_benchmark_queries["search_filters_bm25"] = preprocess_series(
    benchmark_queries["search_infm_params_text"], final_bm25_preprocessing
)
artifact_benchmark_queries["search_text_bm25"] = (
    artifact_benchmark_queries["search_query_bm25"] + " " + artifact_benchmark_queries["search_filters_bm25"]
).str.replace(r"\s+", " ", regex=True).str.strip()
artifact_benchmark_queries["search_query_char_tfidf"] = preprocess_series(
    benchmark_queries["search_query"], final_char_preprocessing
)

train_query_contexts = (
    train_pairs.sort_values("query_group")
    .drop_duplicates("query_group")
    .loc[:, ["query_group", *SEARCH_COLUMNS]]
    .reset_index(drop=True)
)
artifact_train_queries = train_query_contexts.loc[:, ["query_group", "search_category"]].copy()
artifact_train_queries["search_query_bm25"] = preprocess_series(
    train_query_contexts["search_query"], final_bm25_preprocessing
)
artifact_train_queries["search_filters_bm25"] = preprocess_series(
    train_query_contexts["search_infm_params_text"], final_bm25_preprocessing
)
artifact_train_queries["search_text_bm25"] = (
    artifact_train_queries["search_query_bm25"] + " " + artifact_train_queries["search_filters_bm25"]
).str.replace(r"\s+", " ", regex=True).str.strip()
artifact_train_queries["search_query_char_tfidf"] = preprocess_series(
    train_query_contexts["search_query"], final_char_preprocessing
)

items_artifact_path = ARTIFACT_DIR / "benchmark_items_text.parquet"
benchmark_queries_artifact_path = ARTIFACT_DIR / "benchmark_queries_text.parquet"
train_queries_artifact_path = ARTIFACT_DIR / "train_query_contexts_text.parquet"
validation_results_path = ARTIFACT_DIR / "validation_results.csv"
manifest_path = ARTIFACT_DIR / "manifest.json"

artifact_items.to_parquet(items_artifact_path, index=False)
artifact_benchmark_queries.to_parquet(benchmark_queries_artifact_path, index=False)
artifact_train_queries.to_parquet(train_queries_artifact_path, index=False)
pd.DataFrame(results).sort_values("recall@50", ascending=False).to_csv(validation_results_path, index=False)

manifest = {
    "artifact_version": PREPROCESSING_VERSION,
    "created_by": "m1_lexical_retrieval.ipynb",
    "final_experiment": str(best_final_result["experiment"]),
    "bm25_preprocessing": final_bm25_preprocessing,
    "char_tfidf_preprocessing": final_char_preprocessing,
    "preprocessing_specs": PREPROCESSING_SPECS,
    "russian_stopword_count": len(RUSSIAN_STOPWORDS),
    "source_rows": {
        "benchmark_items": len(artifact_items),
        "benchmark_queries": len(artifact_benchmark_queries),
        "unique_train_query_contexts": len(artifact_train_queries),
    },
    "source_files": {
        path.name: {"bytes": path.stat().st_size, "modified_ns": path.stat().st_mtime_ns}
        for path in (TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH)
    },
    "files": [
        items_artifact_path.name,
        benchmark_queries_artifact_path.name,
        train_queries_artifact_path.name,
        validation_results_path.name,
    ],
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

# Lightweight artifact-level verification, not merely a path-exists check.
assert len(pd.read_parquet(items_artifact_path, columns=["item_id", "item_text_bm25"])) == len(candidate_items)
assert len(pd.read_parquet(benchmark_queries_artifact_path, columns=["query_id", "search_text_bm25"])) == len(benchmark_queries)
assert len(pd.read_parquet(train_queries_artifact_path, columns=["query_group", "search_text_bm25"])) == train_pairs["query_group"].nunique()

artifact_seconds = time.perf_counter() - artifact_started
print({
    "artifact_dir": str(ARTIFACT_DIR),
    "artifact_seconds": round(artifact_seconds, 2),
    "final_bm25_preprocessing": final_bm25_preprocessing,
    "final_char_preprocessing": final_char_preprocessing,
    "files": manifest["files"],
})

{'artifact_dir': '/Users/kite/Documents/Карьера/avito-services-candidate-generation/artifacts/text_preprocessing/m1_lexical_best_v1', 'artifact_seconds': 61.6, 'final_bm25_preprocessing': 'stem', 'final_char_preprocessing': 'control', 'files': ['benchmark_items_text.parquet', 'benchmark_queries_text.parquet', 'train_query_contexts_text.parquet', 'validation_results.csv']}


## 9. Results, ClearML logging and decision

The table keeps every candidate source for diagnosis. The retained lexical pipeline is selected only from the RRF final-pipeline rows, so a strong standalone source cannot displace the complementary hybrid without evidence.

In [12]:
results_frame = pd.DataFrame(results).sort_values(["recall@50", "recall@200", "experiment"], ascending=[False, False, True]).reset_index(drop=True)
display(results_frame)

notebook_seconds = time.perf_counter() - NOTEBOOK_STARTED

for _, result in results_frame.iterrows():
    experiment = str(result["experiment"])
    for k in METRIC_KS:
        clearml_logger.report_scalar(
            title=f"Recall@{k}", series=experiment, value=float(result[f"recall@{k}"]), iteration=0
        )
    clearml_logger.report_scalar(title="HitRate@50", series=experiment, value=float(result["hit_rate@50"]), iteration=0)
    clearml_logger.report_scalar(title="Index seconds", series=experiment, value=float(result["index_seconds"]), iteration=0)
    clearml_logger.report_scalar(title="Retrieval seconds", series=experiment, value=float(result["retrieval_seconds"]), iteration=0)
    clearml_logger.report_scalar(title="Peak RSS MB", series=experiment, value=float(result["peak_rss_mb"]), iteration=0)

clearml_logger.report_scalar(title="M1 wall time seconds", series="notebook", value=float(notebook_seconds), iteration=0)
clearml_logger.report_scalar(title="Preprocessing artifact seconds", series="notebook", value=float(artifact_seconds), iteration=0)
clearml_logger.report_scalar(title="Category-114 partition items", series="corpus", value=float((candidate_items["item_category_id"] == 114).sum()), iteration=0)
try:
    clearml_logger.report_table(
        title="M1 validation summary",
        series="E01-E04",
        iteration=0,
        table_plot=results_frame,
    )
    clearml_logger.report_table(
        title="M1 preprocessing ablation",
        series="word_bm25",
        iteration=0,
        table_plot=preprocessing_frame,
    )
    table_log_status = "logged"
except Exception as exc:
    table_log_status = f"table logging skipped: {type(exc).__name__}"

clearml_task.set_parameter("results/best_final_experiment", str(best_final_result["experiment"]))
clearml_task.set_parameter("results/best_final_recall_at_50", float(best_final_result["recall@50"]))
clearml_task.set_parameter("results/best_bm25_preprocessing", final_bm25_preprocessing)
clearml_task.set_parameter("results/best_char_preprocessing", final_char_preprocessing)
clearml_task.set_parameter("artifacts/text_preprocessing_dir", str(ARTIFACT_DIR))
clearml_task.set_parameter("results/notebook_seconds", float(notebook_seconds))
clearml_task.close()

print({
    "best_final_experiment": best_final_result["experiment"],
    "best_final_recall@50": round(float(best_final_result["recall@50"]), 6),
    "best_bm25_preprocessing": final_bm25_preprocessing,
    "best_char_preprocessing": final_char_preprocessing,
    "notebook_seconds": round(notebook_seconds, 2),
    "clearml_table_status": table_log_status,
})

,experiment,representation,index_seconds,retrieval_seconds,total_seconds,features,peak_rss_mb,recall@1,recall@5,recall@10,recall@20,recall@50,recall@200,hit_rate@50,bm25_preprocessing,char_preprocessing
0,E04_rrf_best_bm25_control_char,"RRF(best E04 BM25=stem, control title-char-TFIDF)",0.000000,0.710642,0.710642,0,2773.0,0.025063,0.079991,0.124369,0.191468,0.312988,0.562932,0.323917,stem,control
1,E04_rrf_best_bm25_best_char,"RRF(best E04 BM25=stem, char preprocessing=stem)",0.000000,0.704465,0.704465,0,2773.0,0.023944,0.078768,0.123999,0.188331,0.311417,0.564300,0.322411,stem,stem
2,E04_bm25_all_fields__stem,title + parameters + description | query | pre...,77.999484,8.012227,86.011710,387751,2773.0,0.021704,0.076915,0.126645,0.182526,0.305952,0.561345,0.316384,stem,not_applicable
3,E04_bm25_all_fields__stem_stopwords,title + parameters + description | query | pre...,53.752551,6.957173,60.709725,387689,2773.0,0.021516,0.076915,0.126692,0.183057,0.304867,0.561800,0.315631,stem_stopwords,not_applicable
4,E04_bm25_all_fields__lemma_stopwords,title + parameters + description | query | pre...,56.122796,6.502069,62.624865,396606,2773.0,0.021139,0.076161,0.122742,0.179940,0.297664,0.559289,0.308663,lemma_stopwords,not_applicable
5,E03c_rrf_bm25_all_fields_char_tfidf,"RRF(all-fields-BM25, title-char-TFIDF); reused...",0.000000,1.020952,1.020952,0,2773.0,0.022599,0.076208,0.116538,0.182287,0.297264,0.534450,0.308475,control,control
6,E04_bm25_all_fields__lemma,title + parameters + description | query | pre...,98.937251,7.586830,106.524080,396741,2773.0,0.021045,0.075785,0.124683,0.180333,0.297142,0.560843,0.308098,lemma,not_applicable
7,E01_bm25_title_params_description__query,title + parameters + description | query,46.251047,7.391372,53.642419,595180,2773.0,0.019350,0.074269,0.116720,0.167973,0.284071,0.514539,0.293974,control,not_applicable
8,E04_bm25_all_fields__control,title + parameters + description | query | pre...,46.251047,7.391372,53.642419,595180,2773.0,0.019350,0.074269,0.116720,0.167973,0.284071,0.514539,0.293974,control,not_applicable
9,E04_bm25_all_fields__stopwords,title + parameters + description | query | pre...,46.770035,5.834754,52.604789,594793,2773.0,0.019350,0.074677,0.118404,0.169186,0.283914,0.516028,0.293785,stopwords,not_applicable


{'best_final_experiment': 'E04_rrf_best_bm25_control_char', 'best_final_recall@50': 0.312988, 'best_bm25_preprocessing': 'stem', 'best_char_preprocessing': 'control', 'notebook_seconds': 719.49, 'clearml_table_status': 'logged'}


## M1 conclusion

Use the executed final-pipeline row as the retained lexical baseline. The selected BM25 and char-TFIDF representations are materialized in `artifacts/text_preprocessing/m1_lexical_best_v1/` with a manifest, so M2–M6 do not need to repeat M1 text processing. The directory is intentionally ignored by Git because it contains derived competition data.

**Scope note.** The frozen validation fold has 5,309 queries with `search_category = 114` and one query with category `0`. Category 114 contains 187,336 of 189,212 corpus items, so the hard partition preserves known positives but gives little speed-up for the dominant category. The char-TFIDF comparison intentionally indexes titles only; applying char n-grams to the long parameter field exceeded the one-hour cell limit and is not retained as a feasible M1 baseline. Category 0 has no matching corpus partition and correctly triggers the documented full-corpus fallback.